# **Semantic Web & Linked Open Data with Python**
## บทที่ 08 — OWL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch08/ch08_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://www.w3.org/TR/owl2-primer/
*   https://owlready2.readthedocs.io/

📖 เอกสารประกอบบทนี้: [`08-owl/`](https://github.com/Rathachai/semantic-web/tree/main/08-owl)

---

### Setup

In [1]:
!pip install -q owlready2

In [2]:
import shutil, subprocess

if shutil.which('java') is None:
    print('ไม่พบ Java -- กำลังติดตั้ง (ใช้เวลาราว 30 วินาที บน Colab)...')
    subprocess.run(['apt-get', '-qq', 'install', '-y',
                    'openjdk-17-jre-headless'], check=False)

print('java:', shutil.which('java') or 'ไม่พบ -- reasoning cell ด้านล่างอาจรันไม่ได้')

java: /usr/bin/java


In [3]:
from owlready2 import *
onto = get_ontology('http://example.org/library#')

## Restriction: someValuesFrom (∃) vs allValuesFrom (∀)

In [4]:
with onto:
    class Item(Thing): pass
    class Person(Thing): pass
    class Book(Item): pass
    class DVD(Item): pass

    class borrowedBy(ObjectProperty):
        domain = [Item]; range = [Person]
    class hasItem(ObjectProperty):
        inverse_property = borrowedBy

    # ∃hasItem.Book -- Manchester: BookBorrower EquivalentTo: hasItem some Book
    class BookBorrower(Person):
        equivalent_to = [Person & hasItem.some(Book)]

    # ∀hasItem.Book -- Manchester: BookOnlyBorrower SubClassOf: hasItem only Book
    class BookOnlyBorrower(Person):
        is_a = [hasItem.only(Book)]

print("นิยาม class เสร็จ")

นิยาม class เสร็จ


## Cardinality Restriction

In [5]:
with onto:
    # Manchester: HeavyBorrower EquivalentTo: hasItem min 3
    class HeavyBorrower(Person):
        equivalent_to = [Person & hasItem.min(3)]

print("HeavyBorrower =", HeavyBorrower.equivalent_to)

HeavyBorrower = [library.Person & library.hasItem.min(3, owl.Thing)]


## สร้างข้อมูลทดสอบ

In [6]:
with onto:
    alice = Person("alice")
    b1, b2 = Book("b1"), Book("b2")
    alice.hasItem = [b1, b2]

    bob = Person("bob")
    d1 = DVD("d1")
    bob.hasItem = [d1]

    carol = Person("carol")
    c1, c2, c3 = Book("c1"), Book("c2"), Book("c3")
    carol.hasItem = [c1, c2, c3]

print("สร้าง individual เสร็จ")

สร้าง individual เสร็จ


## Automatic Classification — reasoner จัด class ให้เอง

⚠️ สังเกต: `BookBorrower` ใช้ `equivalent_to` (defined class) จึงจัด individual ให้อัตโนมัติ แต่ `BookOnlyBorrower` ใช้ `is_a` (primitive class) ซึ่งเป็นแค่ **necessary condition** — reasoner จะ**ไม่**จัด alice เข้า class นี้ให้ แม้ alice จะสอดคล้องกับเงื่อนไขก็ตาม นี่คือความต่างระหว่าง primitive กับ defined class

In [7]:
sync_reasoner_hermit([onto], infer_property_values=True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit:/Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/p7/wrfb851s61sg68ywkl0vr3n00000gn/T/tmp227h4lgl -Y
* Owlready2 * HermiT took 0.5123171806335449 seconds
* Owlready * Reparenting library.alice: {library.Person} => {library.BookBorrower}
* Owlready * Reparenting library.carol: {library.Person} => {library.BookBorrower}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [8]:
alice.is_a   # BookBorrower (defined class) ถูกจัดให้อัตโนมัติ

[library.BookBorrower]

In [9]:
bob.is_a     # ไม่ใช่ BookBorrower เพราะยืม DVD ไม่ใช่ Book

[library.Person]

In [10]:
carol.is_a   # ควรเป็น HeavyBorrower ด้วย (ยืม 3 ชิ้น)

[library.BookBorrower]

## Property Inference: inverse property

`hasItem` กับ `borrowedBy` เป็น inverse กัน — ประกาศทางเดียว ได้อีกทางฟรี

In [11]:
b1.borrowedBy   # ไม่เคยประกาศตรง ๆ แต่ reasoner เติมให้จาก inverse_property

[library.alice]

## Unsatisfiable Class

class ที่นิยามขัดแย้งในตัวเองจนไม่มีทางมีสมาชิกได้เลย

In [12]:
with onto:
    class NonItem(Thing):
        equivalent_to = [Not(Item)]
    # class ที่ต้องเป็นทั้ง Item และ NonItem พร้อมกัน -- เป็นไปไม่ได้
    class ImpossibleClass(Item, NonItem):
        pass

sync_reasoner_hermit([onto])
print("ImpossibleClass unsatisfiable:", ImpossibleClass.equivalent_to)
print(list(ImpossibleClass.instances()))

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit:/Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/p7/wrfb851s61sg68ywkl0vr3n00000gn/T/tmpa4gwprzn
* Owlready2 * HermiT took 0.38507914543151855 seconds
* Owlready * Equivalenting: library.ImpossibleClass owl.Nothing
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


ImpossibleClass unsatisfiable: [owl.Nothing]
[]


## Inconsistency: Disjoint Class ที่ถูกละเมิด

`Book` กับ `DVD` เป็น subclass ของ `Item` — ถ้าประกาศว่าทั้งสอง **disjoint** กัน แล้วมี individual ที่เป็นทั้งคู่พร้อมกัน ontology จะพัง

In [13]:
with onto:
    AllDisjoint([Book, DVD])

    weird = Item("weird")
    weird.is_a.append(Book)
    weird.is_a.append(DVD)   # ⚠️ เป็นทั้ง Book และ DVD พร้อมกัน ทั้งที่ disjoint

try:
    sync_reasoner_hermit([onto])
    print("consistent (ไม่ควรเกิด)")
except OwlReadyInconsistentOntologyError:
    print("❌ ตรวจพบ inconsistency ตามคาด")
    print("   weird ถูกประกาศเป็นทั้ง Book และ DVD พร้อมกัน แต่ทั้งสอง disjoint กัน")

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit:/Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/p7/wrfb851s61sg68ywkl0vr3n00000gn/T/tmp94r2812y


❌ ตรวจพบ inconsistency ตามคาด
   weird ถูกประกาศเป็นทั้ง Book และ DVD พร้อมกัน แต่ทั้งสอง disjoint กัน


## 🎯 สรุป

- **`some`** (∃) ต้องมีอย่างน้อยหนึ่งค่า, **`only`** (∀) ทุกค่าต้องเป็นชนิดนี้ (แต่ไม่บังคับว่ามี)
- **Cardinality** (`min`/`max`/`exactly`) จำกัดจำนวนค่า
- reasoner ทำ **automatic classification** ให้เฉพาะ **defined class** (`equivalent_to`) เท่านั้น
  **primitive class** (`is_a`) ไม่ถูกจัดอัตโนมัติแม้ individual จะเข้าเงื่อนไขก็ตาม
- **inverse_property** ทำให้ประกาศทางเดียวแล้วได้อีกทางฟรี
- **unsatisfiable class** คือ class ที่นิยามขัดแย้งในตัวเอง (ไม่ใช่ ontology inconsistent ทั้งก้อน)
- **`AllDisjoint`** ที่ถูกละเมิด (individual เป็นสมาชิกของสอง disjoint class พร้อมกัน) ทำให้ ontology **inconsistent** ทั้งก้อน

ลองเอา `weird.is_a.append(DVD)` ออก แล้วรัน reasoner ใหม่ — ควร consistent ทันที!

---

## จบบทที่ 08 🎉

ไปต่อบทถัดไป: **บทที่ 09 — Well-known Ontologies**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*